# 03 — PV hosting capacity

**Goal:** run the bundled hosting-capacity demonstrator, inspect the network, read the criterion-specific capacity, then verify the evidence.

**Teaching focus:** three-phase PV at bus 675 with the lesson's overvoltage ceiling of 1.05 pu.

**Prediction:** increasing PV should eventually reach the declared overvoltage criterion.

Run the numbered cells in order. The direct OpenDSS section at the end is optional.

In [1]:
#@title 1. Setup — run once
import contextlib, io, urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
try:
    _blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
    _digest = hashlib.sha256(_blob).hexdigest()
    if _digest != _HELPER_SHA256:
        raise ValueError(f"lesson helper hash mismatch (expected {_HELPER_SHA256}, got {_digest})")
    _helper_output = io.StringIO()
    with contextlib.redirect_stdout(_helper_output):
        exec(compile(_blob, "lesson helper", "exec"))
except Exception as exc:
    print("What happened: the pinned lesson helper could not be downloaded or verified.")
    print(f"Details: {type(exc).__name__}: {exc}")
    print("Next: check network access to the public helper URL, then rerun this setup cell.")
    raise SystemExit(1) from None
for _line in _helper_output.getvalue().splitlines():
    if "lesson helpers ready" not in _line.lower():
        print(_line)
print("Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.")

# Compatibility renderer for the currently pinned public wheel; newer wheels use cept.public_notebook.
def display_run_compat(run_dir):
    import html as _html
    from IPython.display import HTML, display
    result = read(Path(run_dir) / 'results.json')
    sld = result.get('sld') or result.get('sld_after') or {}
    nodes, edges = sld.get('nodes') or [], sld.get('edges') or []
    if not nodes:
        display(HTML('<p><strong>CEPT result:</strong> this run does not carry an SLD.</p>'))
        return
    xs, ys = [float(n['x']) for n in nodes], [float(n['y']) for n in nodes]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    dx, dy = max(x1-x0, 1.0), max(y1-y0, 1.0)
    def xy(node):
        return 65 + (float(node['x'])-x0)/dx*870, 45 + (float(node['y'])-y0)/dy*420
    pos = {str(n['id']): xy(n) for n in nodes}
    line_svg = []
    for edge in edges:
        if str(edge.get('src')) in pos and str(edge.get('dst')) in pos:
            a, b = pos[str(edge['src'])], pos[str(edge['dst'])]
            dash = ' stroke-dasharray=\"8 6\"' if edge.get('status') == 'open' else ''
            edge_label = _html.escape(str(edge.get('id', 'branch')))
            line_svg.append(f'<line x1=\"{a[0]:.1f}\" y1=\"{a[1]:.1f}\" x2=\"{b[0]:.1f}\" y2=\"{b[1]:.1f}\" stroke=\"#7a879a\" stroke-width=\"3\"{dash}><title>{edge_label}</title></line>')
    vmin, vmax = float(sld.get('v_min_pu', .95)), float(sld.get('v_max_pu', 1.05))
    bus_svg, rows = [], []
    for node in nodes:
        volts = {int(k): float(v) for k, v in (node.get('v_pu') or {}).items()}
        angles = {int(k): float(v) for k, v in (node.get('angle_deg') or {}).items()}
        values = list(volts.values())
        low, high = any(v < vmin for v in values), any(v > vmax for v in values)
        status = 'NO DATA' if not values else 'OUT' if low and high else 'UNDER' if low else 'OVER' if high else 'OK'
        fill = {'OK':'#e8f5ec','UNDER':'#fff3d9','OVER':'#ffe7e1','OUT':'#f7e7ff','NO DATA':'#eef1f5'}[status]
        stroke = {'OK':'#2f7d4a','UNDER':'#a46700','OVER':'#b8432e','OUT':'#8147a6','NO DATA':'#7b8796'}[status]
        x, y = pos[str(node['id'])]
        phase = lambda p: '—' if p not in volts else f'{volts[p]:.4f} pu' + (f' @ {angles[p]:.2f}°' if p in angles else '')
        tip = _html.escape('Bus '+str(node['id'])+'\nStatus: '+status+'\n'+'\n'.join(f'{label}: {phase(p)}' for p,label in [(1,'A'),(2,'B'),(3,'C')] if p in volts))
        label = _html.escape(str(node['id']))
        bus_svg.append(f'<g tabindex=\"0\"><title>{tip}</title><rect x=\"{x-31:.1f}\" y=\"{y-11:.1f}\" width=\"62\" height=\"22\" rx=\"5\" fill=\"{fill}\" stroke=\"{stroke}\" stroke-width=\"2\"/><text x=\"{x:.1f}\" y=\"{y+4:.1f}\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"700\">{label}</text></g>')
        rows.append('<tr><th>'+label+'</th><td>'+phase(1)+'</td><td>'+phase(2)+'</td><td>'+phase(3)+'</td><td><strong>'+status+'</strong></td></tr>')
    display(HTML('<div style=\"font-family:system-ui,sans-serif\"><h3>Interactive CEPT SLD</h3><p style=\"color:#657187\">Hover or focus a bus for solver-returned values.</p><div style=\"overflow:hidden;border:1px solid #d9dee8;border-radius:10px\"><svg viewBox=\"0 0 1000 510\" style=\"width:100%;height:auto;display:block\">'+''.join(line_svg)+''.join(bus_svg)+'</svg></div><div style=\"overflow-x:auto;margin-top:10px\"><table style=\"border-collapse:collapse;width:100%;min-width:650px\"><thead><tr><th>Bus</th><th>Phase A</th><th>Phase B</th><th>Phase C</th><th>Status</th></tr></thead><tbody>'+''.join(rows)+'</tbody></table></div></div>'))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version
cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


In [2]:
#@title 2. Inputs — hosting-capacity demonstrator
RUN_DIR = WORKSPACE / "runs" / "03-hosting-capacity"
V_MAX_PU = 1.05
DIRECT_SIZES_KW = (0, 1000, 2000)
table(
    ["declared input", "value", "unit"],
    [
        ("network", "IEEE 13-node feeder", "text"),
        ("PV bus", "675", "bus"),
        ("criterion", "overvoltage", "text"),
        ("voltage ceiling", V_MAX_PU, "pu"),
    ],
)

| declared input | value | unit |
| --- | --- | --- |
| network | IEEE 13-node feeder | text |
| PV bus | 675 | bus |
| criterion | overvoltage | text |
| voltage ceiling | 1.05 | pu |


In [3]:
# 3. Run study — same CEPT CLI as a normal terminal
!cept study demo hosting-capacity \
    --network ieee13 \
    --out runs/03-hosting-capacity \
    --force \
    --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the hosting-capacity search and saved the evidence
Saved run          runs\03-hosting-capacity
Case fingerprint   849d2148e0b1 (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\03-hosting-capacity --format text


In [4]:
#@title 4. Explore — SLD and bus status
RUN_DIR = WORKSPACE / "runs" / "03-hosting-capacity"
try:
    from cept.public_notebook import display_run
except ModuleNotFoundError:
    display_run = display_run_compat
display_run(RUN_DIR)

Bus,Phase A,Phase B,Phase C,Status
611,—,—,0.9608 pu @ 115.74°,OK
632,1.0143 pu @ -2.53°,1.0289 pu @ -121.76°,1.0042 pu @ 117.77°,OK
633,1.0113 pu @ -2.59°,1.0270 pu @ -121.81°,1.0015 pu @ 117.76°,OK
634,0.9872 pu @ -3.28°,1.0084 pu @ -122.27°,0.9825 pu @ 117.27°,OK
645,—,1.0197 pu @ -121.94°,1.0023 pu @ 117.79°,OK
646,—,1.0180 pu @ -122.02°,1.0002 pu @ 117.84°,OK
650,0.9999 pu @ -0.01°,1.0000 pu @ -120.01°,0.9999 pu @ 119.99°,OK
652,0.9753 pu @ -5.32°,—,—,OK
670,1.0040 pu @ -3.46°,1.0319 pu @ -121.97°,0.9898 pu @ 117.10°,OK
671,0.9828 pu @ -5.37°,1.0403 pu @ -122.39°,0.9649 pu @ 115.99°,OK


In [5]:
#@title 5. Engineering result — hosting capacity
results = read(RUN_DIR / "results.json")
hosting = results["hosting_capacity"]
item = next(row for row in hosting["items"] if row["bus"].lower() == "675")
table(
    ["quantity", "value", "unit"],
    [
        ("criterion", hosting["criterion"], "text"),
        ("voltage ceiling", hosting["v_max_pu"], "pu"),
        ("baseline maximum voltage", hosting["baseline_v_max_pu"], "pu"),
        ("bus 675 hosting capacity", item["hc_kw"], "kW"),
        ("limit reached", item["limit"], "text"),
    ],
)
assert hosting["criterion"] == "overvoltage"
assert hosting["v_max_pu"] == V_MAX_PU

| quantity | value | unit |
| --- | --- | --- |
| criterion | overvoltage | text |
| voltage ceiling | 1.05 | pu |
| baseline maximum voltage | 1.0426 | pu |
| bus 675 hosting capacity | 1507.6 | kW |
| limit reached | overvoltage | text |


In [6]:
# 6. Verify — check this exact saved run
!cept study verify runs/03-hosting-capacity --format text

CEPT study check: PASSED
----------------------------
Study              Hosting capacity (OpenDSS)
Case fingerprint   849d2148e0b1 (matches the case you ran)

Checked   3 groups, 13 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (2 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     runs\03-hosting-capacity\public-verification.json

For the full check list
  cept study verify runs\03-hosting-capacity --format json


## 7. Interpret

The returned hc_kw is tied to this feeder model and the declared criterion.

**What this proves:** the demonstrator search produced and persisted a criterion-specific result.

**What this does not prove:** utility approval, thermal adequacy, protection adequacy, or project validation.

**Try next:** inspect which assumption or criterion would need to change before treating this as a different hosting-capacity question.

## Optional — direct OpenDSS bracket

The cells below solve 0, 1000, and 2000 kW PV points directly in OpenDSS, then compare that bracket with the persisted CEPT hosting-capacity result.

In [7]:
#@title Under the hood - direct OpenDSS sweep (optional)
MASTER_DSS = ieee13_master()
import opendssdirect as dss

def load_base():
    dss.Basic.ClearAll()
    dss.Basic.DataPath(str(MASTER_DSS.parent))
    dss.Text.Command(f'Redirect "{MASTER_DSS}"')
    dss.Text.Command('CalcVoltageBases')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()

def max_unregulated_voltage():
    excluded = {'sourcebus', '650', 'rg60'}
    names = dss.Circuit.AllNodeNames()
    values = dss.Circuit.AllBusMagPu()
    return max(value for name, value in zip(names, values) if name.split('.')[0].lower() not in excluded)

direct_sweep = {}
for kw in DIRECT_SIZES_KW:
    load_base()
    dss.Text.Command(f'New PVSystem.lesson_pv phases=3 bus1=675.1.2.3 kV=4.16 kVA={max(kw, 1)} Pmpp={kw} irradiance=1 pf=1 %cutin=0.05 %cutout=0.05')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()
    direct_sweep[kw] = max_unregulated_voltage()
os.chdir(WORKSPACE)
print(f"Direct OpenDSS sweep finished: solved {len(direct_sweep)} PV sizes.")


Direct OpenDSS sweep finished: solved 3 PV sizes.


In [8]:
#@title Under the hood — direct sweep readback (optional)
table(['PV size', 'maximum unregulated voltage', 'unit'], [(kw, value, 'pu') for kw, value in direct_sweep.items()])
assert direct_sweep[1000] < V_MAX_PU < direct_sweep[2000]

| PV size | maximum unregulated voltage | unit |
| --- | --- | --- |
| 0 | 1.0426313303211827 | pu |
| 1000 | 1.0476066810380371 | pu |
| 2000 | 1.0521988003106502 | pu |


In [9]:
#@title Compare solver outputs (optional details)
RUN_DIR = WORKSPACE / "runs" / "03-hosting-capacity"
results = read(RUN_DIR / "results.json")
verify_summary = read(RUN_DIR / "public-verification.json")
hosting = results["hosting_capacity"]
item = next(row for row in hosting["items"] if row["bus"].lower() == "675")
table(
    ["field", "value", "unit"],
    [
        ("criterion", hosting["criterion"], "text"),
        ("v_max", hosting["v_max_pu"], "pu"),
        ("baseline_v_max", hosting["baseline_v_max_pu"], "pu"),
        ("bus 675 capacity", item["hc_kw"], "kW"),
        ("bus 675 limit", item["limit"], "text"),
    ],
)
print()
print("CEPT hosting-capacity result")
print("----------------------------")
print(f"Result        {'PASSED' if verify_summary['passed'] else 'Needs attention'}")
print(f"Bus 675 can host about {item['hc_kw']} kW before hitting the voltage limit")
print(f"Limit reached: {item['limit']}")
assert verify_summary["status"] == "PASS"
assert verify_summary["passed"] is True
assert abs(direct_sweep[0] - hosting["baseline_v_max_pu"]) < 1e-3
assert 1000 <= item["hc_kw"] <= 2000
assert hosting["criterion"] == "overvoltage" and hosting["v_max_pu"] == V_MAX_PU


| field | value | unit |
| --- | --- | --- |
| criterion | overvoltage | text |
| v_max | 1.05 | pu |
| baseline_v_max | 1.0426 | pu |
| bus 675 capacity | 1507.6 | kW |
| bus 675 limit | overvoltage | text |

CEPT hosting-capacity result
----------------------------
Result        PASSED
Bus 675 can host about 1507.6 kW before hitting the voltage limit
Limit reached: overvoltage
